In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')
from sklearn.preprocessing import OneHotEncoder,MultiLabelBinarizer,LabelEncoder,MinMaxScaler,StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import ADASYN
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense,Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.models import load_model
from sklearn.utils import class_weight
from sklearn.metrics import classification_report
import numpy as np

In [2]:
data=pd.read_csv('Lending_Loan_Company_Data_Analysis.csv')
data

,Credit_Policy,Loan_Purpose,Interest_Rate,Monthly_Installment,Natural_log_Annual_Income,Debt_Income_Ratio,FICO_Score,Days_with_Credit_Line,Revolving_Balance,Revolving_Utilization,Enquiries_Last_6_Months,Deliquency_Last_2_Yrs,Public_Derogatory_Records,Paid_Status
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,Paid
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,Paid
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,Paid
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,Paid
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9573,0,all_other,0.1461,344.76,12.180755,10.39,672,10474.000000,215372,82.1,2,0,0,Unpaid
9574,0,all_other,0.1253,257.70,11.141862,0.21,722,4380.000000,184,1.1,5,0,0,Unpaid
9575,0,debt_consolidation,0.1071,97.81,10.596635,13.09,687,3450.041667,10036,82.9,8,0,0,Unpaid
9576,0,home_improvement,0.1600,351.58,10.819778,19.18,692,1800.000000,0,3.2,5,0,0,Unpaid


# Feature Engineering

## Check for Collinearity B/W the Columns

In [54]:
for i in data_num.columns:
    corr=data_num.corr()
    corr[i].sort_values(ascending=False)
    print(corr)

                           Credit_Policy  Interest_Rate  Monthly_Installment  \
Credit_Policy                   1.000000      -0.294089             0.058770   
Interest_Rate                  -0.294089       1.000000             0.276140   
Monthly_Installment             0.058770       0.276140             1.000000   
Natural_log_Annual_Income       0.034906       0.056383             0.448102   
Debt_Income_Ratio              -0.090901       0.220006             0.050202   
FICO_Score                      0.348319      -0.714821             0.086039   
Days_with_Credit_Line           0.099026      -0.124022             0.183297   
Revolving_Balance              -0.187518       0.092527             0.233625   
Revolving_Utilization          -0.104095       0.464837             0.081356   
Enquiries_Last_6_Months        -0.535511       0.202780            -0.010419   
Deliquency_Last_2_Yrs          -0.076318       0.156079            -0.004368   
Public_Derogatory_Records      -0.054243

### Since, Most of the Columns have the Scores less Than 0.5 between them, There is no need to drop off the columns

# Model Building

## 1. Encode the Labels

In [58]:
data_model=data.copy()

In [59]:
le=LabelEncoder()

In [60]:
data_model

,Credit_Policy,Loan_Purpose,Interest_Rate,Monthly_Installment,Natural_log_Annual_Income,Debt_Income_Ratio,FICO_Score,Days_with_Credit_Line,Revolving_Balance,Revolving_Utilization,Enquiries_Last_6_Months,Deliquency_Last_2_Yrs,Public_Derogatory_Records,Paid_Status
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,Paid
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,Paid
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,Paid
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,Paid
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9573,0,all_other,0.1461,344.76,12.180755,10.39,672,10474.000000,215372,82.1,2,0,0,Unpaid
9574,0,all_other,0.1253,257.70,11.141862,0.21,722,4380.000000,184,1.1,5,0,0,Unpaid
9575,0,debt_consolidation,0.1071,97.81,10.596635,13.09,687,3450.041667,10036,82.9,8,0,0,Unpaid
9576,0,home_improvement,0.1600,351.58,10.819778,19.18,692,1800.000000,0,3.2,5,0,0,Unpaid


In [61]:
data['Loan_Purpose'].unique()

array(['debt_consolidation', 'credit_card', 'all_other',
       'home_improvement', 'small_business', 'major_purchase',
       'educational'], dtype=object)

In [62]:
data_model['Loan_Purpose'].unique()

array(['debt_consolidation', 'credit_card', 'all_other',
       'home_improvement', 'small_business', 'major_purchase',
       'educational'], dtype=object)

In [63]:
data

,Credit_Policy,Loan_Purpose,Interest_Rate,Monthly_Installment,Natural_log_Annual_Income,Debt_Income_Ratio,FICO_Score,Days_with_Credit_Line,Revolving_Balance,Revolving_Utilization,Enquiries_Last_6_Months,Deliquency_Last_2_Yrs,Public_Derogatory_Records,Paid_Status
0,1,debt_consolidation,0.1189,829.10,11.350407,19.48,737,5639.958333,28854,52.1,0,0,0,Paid
1,1,credit_card,0.1071,228.22,11.082143,14.29,707,2760.000000,33623,76.7,0,0,0,Paid
2,1,debt_consolidation,0.1357,366.86,10.373491,11.63,682,4710.000000,3511,25.6,1,0,0,Paid
3,1,debt_consolidation,0.1008,162.34,11.350407,8.10,712,2699.958333,33667,73.2,1,0,0,Paid
4,1,credit_card,0.1426,102.92,11.299732,14.97,667,4066.000000,4740,39.5,0,1,0,Paid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9573,0,all_other,0.1461,344.76,12.180755,10.39,672,10474.000000,215372,82.1,2,0,0,Unpaid
9574,0,all_other,0.1253,257.70,11.141862,0.21,722,4380.000000,184,1.1,5,0,0,Unpaid
9575,0,debt_consolidation,0.1071,97.81,10.596635,13.09,687,3450.041667,10036,82.9,8,0,0,Unpaid
9576,0,home_improvement,0.1600,351.58,10.819778,19.18,692,1800.000000,0,3.2,5,0,0,Unpaid


## Description of Labels

### 1. Loan Purpose
#### * debt_consolidation - 2
#### * credit_card - 1
#### * all_other - 0
#### * home_improvement -4
#### * small_business - 6
#### * major_purchase - 5
#### * educational - 3

### 2. Paid Status
#### * Paid - 0
#### * Unpaid - 1

In [65]:
data_model['Installment_to_Income'] = data_model['Monthly_Installment'] / (data['Natural_log_Annual_Income'] / 12)
data_model['Interest_Installment'] = data_model['Interest_Rate'] * data_model['Monthly_Installment']
data_model['Revolving_to_Income'] = data_model['Revolving_Balance'] / data_model['Natural_log_Annual_Income']
data_model['Credit_Length_Years'] = data_model['Days_with_Credit_Line'] / 365
data_model['Risk_Score'] = data_model['Interest_Rate'] / data_model['FICO_Score']

In [66]:
x=data_model.drop(['Paid_Status','Natural_log_Annual_Income'],axis=1)
y=le.fit_transform(data_model['Paid_Status'])

In [67]:
xtrain,xtemp,ytrain,ytemp=train_test_split(x,y,test_size=0.3,random_state=42)
xeval,xtest,yeval,ytest=train_test_split(xtemp,ytemp,test_size=0.5,random_state=42)

In [68]:

cat_cols = ['Credit_Policy', 'Loan_Purpose', 'Enquiries_Last_6_Months' ,'Deliquency_Last_2_Yrs','Public_Derogatory_Records']

# Outlier numeric
robust_cols = [
    'Interest_Rate',
    'Monthly_Installment',
    'FICO_Score',
    'Days_with_Credit_Line',
    'Revolving_Balance',
    'Installment_to_Income',
    'Interest_Installment'
    
]

# Remaining numeric
standard_cols = [col for col in xtrain.columns 
                 if col not in robust_cols + cat_cols]



In [69]:

preprocessor = ColumnTransformer(
    transformers=[
        ('robust', RobustScaler(), robust_cols),
        ('standard', StandardScaler(), standard_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)   
    ]
)

# Step 1: Fit scaler on original train
X_train_scaled = preprocessor.fit_transform(xtrain)

# Step 2: Apply same transform
X_eval = preprocessor.transform(xeval)
X_test = preprocessor.transform(xtest)

# Step 3: Apply ADASYN AFTER scaling
adasyn = ADASYN(sampling_strategy=0.5, random_state=42)
X_train_res_Ada, y_train_res = adasyn.fit_resample(X_train_scaled, ytrain)

In [70]:
model = Sequential()

# Input Layer + Hidden Layer 1
model.add(Dense(64, activation='relu', input_dim=X_train_res_Ada.shape[1]))
model.add(Dropout(0.3))

# Hidden Layer 2
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))

# Hidden Layer 3
model.add(Dense(16, activation='relu'))

# Output Layer
model.add(Dense(1, activation='sigmoid'))  # Binary classification

# Compile
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['Precision', 'Recall']
)

# Early Stopping (VERY IMPORTANT)
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=20,
    restore_best_weights=True
)

# Train
history = model.fit(
    X_train_res_Ada, y_train_res,
    validation_data=(X_eval, yeval),
    
    epochs=200,
    batch_size=32,
    callbacks=[early_stop]
    
)

Epoch 1/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - Precision: 0.3549 - Recall: 0.2265 - loss: 0.6726 - val_Precision: 0.4130 - val_Recall: 0.0809 - val_loss: 0.4875
Epoch 2/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.5333 - Recall: 0.0911 - loss: 0.6086 - val_Precision: 0.5263 - val_Recall: 0.0851 - val_loss: 0.4877
Epoch 3/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.5617 - Recall: 0.1449 - loss: 0.6060 - val_Precision: 0.4359 - val_Recall: 0.0723 - val_loss: 0.4853
Epoch 4/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.5719 - Recall: 0.1691 - loss: 0.6030 - val_Precision: 0.3684 - val_Recall: 0.1191 - val_loss: 0.4938
Epoch 5/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.5928 - Recall: 0.2225 - loss: 0.5905 - val_Precision: 0.3117 - val_Recall: 0.2043 - val_loss: 0.5058
Epoch 6/200
268/268 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - Precision: 0.6120 - Recall: 0.2760 - loss: 0.5879 - val_Precision: 0.3185 - val_Recall: 0.1830 - val

In [71]:
loss, precision, recall = model.evaluate(X_eval, yeval)
print("Loss:", loss)
print("Precision:", precision)
print("Recall:", recall)

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - Precision: 0.2562 - Recall: 0.1619 - loss: 0.4889     
Loss: 0.4822766184806824
Precision: 0.27819550037384033
Recall: 0.15744680166244507


In [138]:
ypred=model.predict(X_eval)

ypred_t = (model.predict(X_eval) > 0.45).astype(int)

print(classification_report(yeval, ypred_t))


45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.86      0.87      0.86      1202
           1       0.28      0.26      0.27       235

    accuracy                           0.77      1437
   macro avg       0.57      0.56      0.56      1437
weighted avg       0.76      0.77      0.77      1437



In [139]:
ypred_Test=model.predict(X_test)

ypred_test = (model.predict(X_test) > 0.45).astype(int)

print(classification_report(ytest, ypred_test))

45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step
45/45 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step
              precision    recall  f1-score   support

           0       0.86      0.86      0.86      1206
           1       0.29      0.29      0.29       231

    accuracy                           0.77      1437
   macro avg       0.58      0.58      0.58      1437
weighted avg       0.77      0.77      0.77      1437

